In [7]:
#DLS

class PuzzleState:
    def __init__(self, board, parent=None, move=None, depth=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.depth = depth

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(self.board)

    def find_blank(self):
        return self.board.index(0)

    def get_neighbors(self):
        neighbors = []
        blank_index = self.find_blank()
        row, col = divmod(blank_index, 3)

        moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

        for dr, dc, move_name in moves:
            new_row, new_col = row + dr, col + dc

            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_blank_index = new_row * 3 + new_col

                new_board = list(self.board)
                new_board[blank_index], new_board[new_blank_index] = new_board[new_blank_index], new_board[blank_index]

                neighbors.append(PuzzleState(tuple(new_board), self, move_name, self.depth + 1))

        return neighbors

def solve_dfs(start_state, goal_state, max_depth=20):
    """Solves the 8-puzzle using Depth-Limited DFS."""
    stack = [PuzzleState(start_state)]
    visited = set()

    while stack:
        current = stack.pop()

        if current.board == goal_state:
            return reconstruct_path(current)

        if current.depth < max_depth:
            visited.add(current.board)

            for neighbor in reversed(current.get_neighbors()):
                if neighbor.board not in visited and neighbor not in stack:
                    stack.append(neighbor)

    return None

def reconstruct_path(state):
    path = []
    while state.parent is not None:
        path.append((state.move, state.board))
        state = state.parent
    return path[::-1]

def print_board(board):
    for i in range(0, 9, 3):
        print(f"{board[i]} {board[i+1]} {board[i+2]}")
    print("-" * 5)

if __name__ == "__main__":
    initial_configuration = (2,8,3,
                             1,6,4,
                             7,0,5)

    goal_configuration = (1, 2, 3,
                           8,0,4,
                           7,6,5)

    print("Initial Board:")
    print_board(initial_configuration)

    solution = solve_dfs(initial_configuration, goal_configuration, max_depth=15)

    if solution:
        print(f"Solution found in {len(solution)} moves!\n")
        for step, (move, board) in enumerate(solution, 1):
            print(f"Step {step}: Move Blank '{move}'")
            print_board(board)
    else:
        print("No solution found within the specified depth limit.")

Initial Board:
2 8 3
1 6 4
7 0 5
-----
Solution found in 5 moves!

Step 1: Move Blank 'Up'
2 8 3
1 0 4
7 6 5
-----
Step 2: Move Blank 'Up'
2 0 3
1 8 4
7 6 5
-----
Step 3: Move Blank 'Left'
0 2 3
1 8 4
7 6 5
-----
Step 4: Move Blank 'Down'
1 2 3
0 8 4
7 6 5
-----
Step 5: Move Blank 'Right'
1 2 3
8 0 4
7 6 5
-----


In [9]:
# Iterative Deepening

# Function to print the 3x3 matrix layout
def print_matrix(state):
    for i in range(0, 9, 3):
        print(f"  {state[i]} {state[i+1]} {state[i+2]}")
    print("-" * 12)

# Generate valid adjacent states and their moving directions
def get_neighbors(state):
    neighbors = []
    idx = state.index(0)  # Find the empty space
    r, c = idx // 3, idx % 3

    moves = [
        (-1, 0, "UP"),
        (1, 0, "DOWN"),
        (0, -1, "LEFT"),
        (0, 1, "RIGHT")
    ]

    for dr, dc, direction in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            n_idx = nr * 3 + nc
            new_state = list(state)
            new_state[idx], new_state[n_idx] = new_state[n_idx], new_state[idx]
            neighbors.append((tuple(new_state), direction))
    return neighbors

# Depth-Limited Search helper
def dls(state, goal, depth, path_states, path_dirs, visited):
    if state == goal:
        return path_states, path_dirs
    if depth <= 0:
        return None

    visited.add(state)
    for neighbor, direction in get_neighbors(state):
        if neighbor not in visited:
            result = dls(neighbor, goal, depth - 1, path_states + [neighbor], path_dirs + [direction], visited)
            if result is not None:
                return result
    visited.remove(state)
    return None

# Core Iterative Deepening Search loop
def ids_8_puzzle(start, goal):
    depth = 0
    while True:
        visited = set()
        result = dls(start, goal, depth, [start], [], visited)
        if result is not None:
            return result
        depth += 1

# --- User Input Helper ---
def get_user_state(prompt, default_state):
    print(f"\n{prompt} (Press Enter for default: {list(default_state)}):")
    user_in = input().strip()
    if not user_in:
        return default_state
    return tuple(map(int, user_in.split()))

# --- Run Execution ---
default_initial = (2, 8, 3, 1, 6, 4, 7, 0, 5)
default_goal    = (1, 2, 3, 8, 0, 4, 7, 6, 5)

print("--- 8 PUZZLE IDS SOLVER ---")
initial_state = get_user_state("Enter 9 numbers for Initial Configuration", default_initial)
goal_state    = get_user_state("Enter 9 numbers for Goal Configuration", default_goal)

print("\nRunning Iterative Deepening Search...\n")
states, moves = ids_8_puzzle(initial_state, goal_state)

if states:
    print("Initial Matrix Configuration:")
    print_matrix(states[0])

    for i in range(len(moves)):
        print(f"Action: Move Blank Space {moves[i]}")
        print_matrix(states[i+1])
else:
    print("No solution found.")


--- 8 PUZZLE IDS SOLVER ---

Enter 9 numbers for Initial Configuration (Press Enter for default: [2, 8, 3, 1, 6, 4, 7, 0, 5]):


Enter 9 numbers for Goal Configuration (Press Enter for default: [1, 2, 3, 8, 0, 4, 7, 6, 5]):


Running Iterative Deepening Search...

Initial Matrix Configuration:
  2 8 3
  1 6 4
  7 0 5
------------
Action: Move Blank Space UP
  2 8 3
  1 0 4
  7 6 5
------------
Action: Move Blank Space UP
  2 0 3
  1 8 4
  7 6 5
------------
Action: Move Blank Space LEFT
  0 2 3
  1 8 4
  7 6 5
------------
Action: Move Blank Space DOWN
  1 2 3
  0 8 4
  7 6 5
------------
Action: Move Blank Space RIGHT
  1 2 3
  8 0 4
  7 6 5
------------
